# Project Phase 1 — Parts Covered by Lab 3

This notebook solves **only** the parts of Project Phase 1 that were discussed in Lab 3:
- Data Exploration (shapes, class labels, target statistics)
- Class Distribution Plotting
- Ridge Classification (cross-validated) for classification datasets
- Ridge Regression (cross-validated) for regression datasets
- Performance metrics (Accuracy, NRMSE)

**NOT included here:** Neural Network variants (not covered in Lab 3).

---

### Legend — What Comes From Where

Throughout this notebook, every concept is tagged:

| Tag | Meaning |
|---|---|
| 🟢 **FROM LAB 3** | Directly uses a concept/function demonstrated in Lab 3 |
| 🔵 **NEW CONCEPT** | Something added beyond Lab 3 to complete the project |

## 1. Loading Dependencies

🟢 **FROM LAB 3** — `numpy`, `pandas`, `matplotlib`, `sklearn.preprocessing`, `sklearn.linear_model`, `train_test_split` are all imported in Lab 3.

In [ ]:
# 🟢 FROM LAB 3 — These exact imports appear in Lab 3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import preprocessing, linear_model
from sklearn.model_selection import train_test_split

# 🔵 NEW CONCEPT — RidgeClassifierCV and RidgeCV
# Lab 3 uses linear_model.Ridge (manual alpha search) and linear_model.LogisticRegression.
# The project requires CROSS-VALIDATED versions that automatically find the best alpha.
# RidgeClassifierCV: a Ridge-based classifier with built-in cross-validation over alphas.
# RidgeCV: a Ridge regressor with built-in cross-validation over alphas.
# These save you from writing the manual alpha loop shown in Lab 3's Boston dataset section.
from sklearn.linear_model import RidgeClassifierCV, RidgeCV

# 🔵 NEW CONCEPT — LabelEncoder
# Lab 3 only worked with datasets that already had numerical labels.
# The project handout warns: "some data may not have numerical class labels.
# You have to change them to numerical before you can work with them."
# LabelEncoder converts string labels like ['cat','dog'] to [0, 1].
from sklearn.preprocessing import LabelEncoder

# 🔵 NEW CONCEPT — os module for file path handling
import os

## 2. Setup — Data Path Configuration

🔵 **NEW CONCEPT** — Lab 3 loaded CSVs or sklearn built-in datasets. The project provides `.npy` files (NumPy binary arrays) organized in folders. You load them with `np.load()`.

**Update `DATA_ROOT` below to point to wherever you extracted the project data.**

In [ ]:
# 🔵 NEW — Set this to the root folder containing all dataset subfolders
DATA_ROOT = "./data/"  # <-- CHANGE THIS to your actual path

# Pick 2 classification + 2 regression datasets (handout recommends AppliancesEnergy)
CLASSIFICATION_DATASETS = ["AbnormalHeartbeat", "SelfRegulationSCP2"]  # <-- Pick any 2 from the 4
REGRESSION_DATASETS     = ["AppliancesEnergy", "BIDMC32RR"]            # <-- Pick any 2 from the 4

# 🔵 NEW — Helper function to load .npy data from the project's folder structure
# Each dataset folder is expected to have: x_train.npy, y_train.npy, x_test.npy, y_test.npy
def load_dataset(name):
    """Load x_train, y_train, x_test, y_test from a named dataset folder."""
    folder = os.path.join(DATA_ROOT, name)
    x_train = np.load(os.path.join(folder, "x_train.npy"))
    y_train = np.load(os.path.join(folder, "y_train.npy"))
    x_test  = np.load(os.path.join(folder, "x_test.npy"))
    y_test  = np.load(os.path.join(folder, "y_test.npy"))
    return x_train, y_train, x_test, y_test

---
# Part A — Classification
---

## 3. Explore Classification Data

Fill the table: **Dataset name | #Samples (Train) | #Samples (Test) | #Classes | Class labels**

🟢 **FROM LAB 3** — Checking `.shape`, printing data (same as `X.shape`, `y.shape` in Lab 3).

🔵 **NEW CONCEPT** — `np.unique()` to find unique class labels and their counts. Lab 3 never needed this because it worked with predefined datasets where classes were known.

In [ ]:
# 🔵 NEW — LabelEncoder instances (one per dataset, in case labels are strings)
label_encoders = {}

for ds_name in CLASSIFICATION_DATASETS:
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")
    
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    
    # 🟢 FROM LAB 3 — Printing shapes (same pattern as Lab 3: print(X.shape), print(y.shape))
    print(f"x_train shape: {x_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"x_test  shape: {x_test.shape}")
    print(f"y_test  shape: {y_test.shape}")
    
    # 🔵 NEW — np.unique() to extract class labels and counts
    # Lab 3 never needed to discover classes; here we must find them ourselves.
    all_labels = np.concatenate([y_train, y_test])
    unique_labels, counts = np.unique(all_labels, return_counts=True)
    
    print(f"#Samples (Train): {x_train.shape[0]}")
    print(f"#Samples (Test) : {x_test.shape[0]}")
    print(f"#Classes        : {len(unique_labels)}")
    print(f"Class labels    : {unique_labels}")
    
    # 🔵 NEW — Encode non-numerical labels if needed
    # The project handout says: "some data may not have numerical class labels."
    if y_train.dtype.kind in ('U', 'S', 'O'):  # string types
        print(f"  ⚠ Labels are non-numeric ({y_train.dtype}). Encoding to integers...")
        le = LabelEncoder()
        le.fit(all_labels)
        label_encoders[ds_name] = le
        print(f"  Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
    else:
        print(f"  ✓ Labels are already numeric.")

## 4. Plot Class Distribution

🟢 **FROM LAB 3** — `matplotlib.pyplot` for plotting (Lab 3 uses `plt.scatter`, `plt.plot`, `plt.grid`).

🔵 **NEW CONCEPT** — `plt.bar()` for bar charts. Lab 3 only used scatter and line plots. A bar chart is the natural choice for showing how many samples belong to each class.

In [ ]:
for ds_name in CLASSIFICATION_DATASETS:
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    all_labels = np.concatenate([y_train, y_test])
    
    # Encode if needed
    if ds_name in label_encoders:
        all_labels = label_encoders[ds_name].transform(all_labels)
    
    unique_labels, counts = np.unique(all_labels, return_counts=True)
    
    # 🔵 NEW — Bar chart for class distribution (Lab 3 never plots distributions)
    plt.figure(figsize=(6, 4))
    plt.bar([str(l) for l in unique_labels], counts, color='steelblue', edgecolor='black')
    plt.title(f"Class Distribution — {ds_name}")
    plt.xlabel("Class Label")
    plt.ylabel("Number of Samples")
    plt.grid(axis='y')  # 🟢 FROM LAB 3 — plt.grid('on') pattern
    plt.tight_layout()
    plt.show()
    
    # Also print train vs test split per class
    print(f"\n{ds_name} — Train split:")
    y_tr = y_train if ds_name not in label_encoders else label_encoders[ds_name].transform(y_train)
    for lbl in unique_labels:
        print(f"  Class {lbl}: {np.sum(y_tr == lbl)} samples")
    print()

---
# Part B — Regression
---

## 5. Explore Regression Data

Fill the table: **Dataset name | #Samples (Train) | #Samples (Test) | Target mean | Target var**

🟢 **FROM LAB 3** — `.shape` for sample counts.

🔵 **NEW CONCEPT** — `np.mean()` and `np.var()` on the target. Lab 3 computed MSE (which uses mean), but never explicitly reported target statistics for exploration. The project wants you to understand your target distribution before modeling.

In [ ]:
for ds_name in REGRESSION_DATASETS:
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")
    
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    
    # 🟢 FROM LAB 3 — shape inspection
    print(f"x_train shape: {x_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"x_test  shape: {x_test.shape}")
    print(f"y_test  shape: {y_test.shape}")
    
    print(f"\n#Samples (Train): {x_train.shape[0]}")
    print(f"#Samples (Test) : {x_test.shape[0]}")
    
    # 🔵 NEW — Target statistics (mean, variance)
    # Lab 3 never explicitly computed these for exploration.
    all_y = np.concatenate([y_train, y_test])
    print(f"Target mean     : {np.mean(all_y):.4f}")
    print(f"Target variance : {np.var(all_y):.4f}")

---
# Part C — Cross-Validated Ridge Classifier
---

### What's from Lab 3 vs. What's New

| Concept | Source |
|---|---|
| Ridge regression (L2 penalty, alpha parameter) | 🟢 Lab 3 — `linear_model.Ridge(alpha=...)` |
| Manual alpha grid search loop | 🟢 Lab 3 — the `for alpha in alpha_test:` loop on Boston data |
| MinMax normalization | 🟢 Lab 3 — `preprocessing.minmax_scale(X)` |
| Accuracy = `np.mean(y == y_pred)*100` | 🟢 Lab 3 — Breast Cancer logistic regression section |
| `RidgeClassifierCV` (auto cross-validation) | 🔵 **NEW** — replaces the manual loop |
| `class_weight` parameter | 🔵 **NEW** — project specifically asks you to explore this |

#### 🔵 What is `RidgeClassifierCV`?
In Lab 3, you manually looped over alpha values:
```python
alpha_test = [0, 0.01, 0.1, 1, 10, 100, 1000, 10000]
for alpha in alpha_test:
    model = linear_model.Ridge(alpha=alpha)
    ...
```
`RidgeClassifierCV` does this **automatically** using internal cross-validation. You give it a list of alphas, and it picks the best one. No need for a separate validation set!

#### 🔵 What is `class_weight`?
Lab 3 never dealt with **imbalanced classes** (where one class has many more samples than another). If class A has 900 samples and class B has 100, the model may learn to always predict A. Setting `class_weight='balanced'` tells the model to penalize mistakes on the minority class more heavily, proportional to how rare it is.

In [ ]:
# 🟢 FROM LAB 3 — Alpha candidates (same idea as Lab 3's alpha_test list)
alphas = [0.01, 0.1, 1, 10, 100, 1000, 10000]

classification_results = []

for ds_name in CLASSIFICATION_DATASETS:
    print(f"\n{'='*60}")
    print(f"Classification: {ds_name}")
    print(f"{'='*60}")
    
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    
    # 🔵 NEW — Encode labels if they are non-numeric
    if ds_name in label_encoders:
        le = label_encoders[ds_name]
        y_train = le.transform(y_train)
        y_test  = le.transform(y_test)
    
    # 🟢 FROM LAB 3 — MinMax normalization
    # Lab 3: X_norm = preprocessing.minmax_scale(X)
    # IMPORTANT: Fit scaler on train only, then transform both train and test.
    # Lab 3 normalized the entire dataset before splitting — here we do it properly.
    # 🔵 NEW NUANCE — Lab 3 called minmax_scale on ALL data before splitting.
    #   That technically leaks test info into training (the scaler sees test data).
    #   The correct way is to fit on train, then transform test with the same scaler.
    scaler = preprocessing.MinMaxScaler()
    x_train_norm = scaler.fit_transform(x_train)
    x_test_norm  = scaler.transform(x_test)
    
    # 🔵 NEW — RidgeClassifierCV with built-in cross-validation
    # This replaces Lab 3's manual "for alpha in alpha_test" loop.
    # Internally, it splits x_train into folds and tests each alpha.
    model = RidgeClassifierCV(alphas=alphas)
    model.fit(x_train_norm, y_train)
    
    best_alpha = model.alpha_
    print(f"Best alpha (via cross-validation): {best_alpha}")
    
    # 🟢 FROM LAB 3 — Accuracy computation
    # Lab 3: acc_training = np.mean(y_train == model.predict(X_train))*100
    y_pred_train = model.predict(x_train_norm)
    y_pred_test  = model.predict(x_test_norm)
    
    acc_train = np.mean(y_train == y_pred_train) * 100
    acc_test  = np.mean(y_test  == y_pred_test)  * 100
    
    print(f"Training accuracy (%): {acc_train:.2f}")
    print(f"Testing  accuracy (%): {acc_test:.2f}")
    
    classification_results.append({
        'Dataset': ds_name,
        'Train Acc': f"{acc_train:.2f}",
        'Test Acc': f"{acc_test:.2f}",
        'Alpha': best_alpha
    })

### 🔵 NEW — Explore `class_weight='balanced'`

The project handout says: *"Explore the class_weight attribute for RidgeClassifierCV. See if adding class weights improves the accuracy."*

This is entirely new — Lab 3 never discusses class imbalance or class weights.

**How it works:** When `class_weight='balanced'`, each sample's influence is scaled inversely proportional to its class frequency:
$$w_c = \frac{n_{\text{samples}}}{n_{\text{classes}} \times n_{\text{samples in class } c}}$$

So rare classes get higher weight and the model can't just ignore them.

In [ ]:
# 🔵 ENTIRELY NEW — class_weight exploration
classification_results_weighted = []

for ds_name in CLASSIFICATION_DATASETS:
    print(f"\n{'='*60}")
    print(f"Classification (class_weight='balanced'): {ds_name}")
    print(f"{'='*60}")
    
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    
    if ds_name in label_encoders:
        le = label_encoders[ds_name]
        y_train = le.transform(y_train)
        y_test  = le.transform(y_test)
    
    scaler = preprocessing.MinMaxScaler()
    x_train_norm = scaler.fit_transform(x_train)
    x_test_norm  = scaler.transform(x_test)
    
    # 🔵 NEW — class_weight='balanced' tells the classifier to upweight rare classes
    model_weighted = RidgeClassifierCV(alphas=alphas, class_weight='balanced')
    model_weighted.fit(x_train_norm, y_train)
    
    best_alpha_w = model_weighted.alpha_
    print(f"Best alpha (balanced): {best_alpha_w}")
    
    y_pred_train_w = model_weighted.predict(x_train_norm)
    y_pred_test_w  = model_weighted.predict(x_test_norm)
    
    acc_train_w = np.mean(y_train == y_pred_train_w) * 100
    acc_test_w  = np.mean(y_test  == y_pred_test_w)  * 100
    
    print(f"Training accuracy (%): {acc_train_w:.2f}")
    print(f"Testing  accuracy (%): {acc_test_w:.2f}")
    
    classification_results_weighted.append({
        'Dataset': ds_name,
        'Train Acc (balanced)': f"{acc_train_w:.2f}",
        'Test Acc (balanced)': f"{acc_test_w:.2f}",
        'Alpha': best_alpha_w
    })

### Classification Results Summary Table

In [ ]:
print("\n" + "="*70)
print("CLASSIFICATION RESULTS — Without class_weight")
print("="*70)
# 🟢 FROM LAB 3 — Using pandas DataFrames for display (Lab 3 uses df.head())
df_cls = pd.DataFrame(classification_results)
print(df_cls.to_string(index=False))

print("\n" + "="*70)
print("CLASSIFICATION RESULTS — With class_weight='balanced'")
print("="*70)
df_cls_w = pd.DataFrame(classification_results_weighted)
print(df_cls_w.to_string(index=False))

---
# Part D — Cross-Validated Ridge Regressor
---

### What's from Lab 3 vs. What's New

| Concept | Source |
|---|---|
| Ridge regression (L2 penalty) | 🟢 Lab 3 — `linear_model.Ridge(alpha=...)` on Boston data |
| MinMax normalization | 🟢 Lab 3 — `preprocessing.minmax_scale(X)` |
| MSE = `np.mean((y - y_pred)**2)` | 🟢 Lab 3 — used throughout |
| `RidgeCV` (auto cross-validation) | 🔵 **NEW** — replaces the manual alpha loop |
| NRMSE (Normalized RMSE) | 🔵 **NEW** — the project metric |

#### 🔵 What is NRMSE?
Lab 3 only computed **MSE** (Mean Squared Error):
```python
mse = np.mean((y - y_pred)**2)
```

The project asks for **NRMSE** (Normalized Root Mean Squared Error), scaled by standard deviation:

$$\text{NRMSE} = \frac{\text{RMSE}}{\sigma_y} = \frac{\sqrt{\text{MSE}}}{\text{std}(y)}$$

Why? MSE depends on the scale of your target. If one dataset's target ranges 0–10 and another's ranges 0–10000, their MSE values are not comparable. NRMSE normalizes by the spread of the data, making it **unit-free** and **comparable across datasets**.

In [ ]:
# 🔵 NEW — NRMSE helper function
# Lab 3 only had MSE. NRMSE = sqrt(MSE) / std(y_true)
def compute_nrmse(y_true, y_pred):
    """Standard-deviation-scaled Normalized Root Mean Squared Error."""
    mse  = np.mean((y_true - y_pred) ** 2)       # 🟢 This part is from Lab 3
    rmse = np.sqrt(mse)                           # 🔵 NEW — take the square root
    nrmse = rmse / np.std(y_true)                 # 🔵 NEW — divide by std of ground truth
    return nrmse

In [ ]:
# 🟢 FROM LAB 3 — Alpha candidates
alphas_reg = [0.01, 0.1, 1, 10, 100, 1000, 10000]

regression_results = []

for ds_name in REGRESSION_DATASETS:
    print(f"\n{'='*60}")
    print(f"Regression: {ds_name}")
    print(f"{'='*60}")
    
    x_train, y_train, x_test, y_test = load_dataset(ds_name)
    
    # 🟢 FROM LAB 3 — MinMax normalization
    scaler = preprocessing.MinMaxScaler()
    x_train_norm = scaler.fit_transform(x_train)
    x_test_norm  = scaler.transform(x_test)
    
    # 🔵 NEW — RidgeCV (cross-validated ridge regressor)
    # Same idea as RidgeClassifierCV but for regression.
    # In Lab 3, you manually looped over alphas with linear_model.Ridge.
    # RidgeCV does this automatically with internal cross-validation.
    model = RidgeCV(alphas=alphas_reg)
    model.fit(x_train_norm, y_train)
    
    best_alpha = model.alpha_
    print(f"Best alpha (via cross-validation): {best_alpha}")
    
    # 🟢 FROM LAB 3 — Predict and compute error
    y_pred_train = model.predict(x_train_norm)
    y_pred_test  = model.predict(x_test_norm)
    
    # 🟢 FROM LAB 3 — MSE (for reference)
    mse_train = np.mean((y_train - y_pred_train) ** 2)
    mse_test  = np.mean((y_test  - y_pred_test)  ** 2)
    print(f"MSE  (train): {mse_train:.4f}")
    print(f"MSE  (test) : {mse_test:.4f}")
    
    # 🔵 NEW — NRMSE (the project's required metric)
    nrmse_train = compute_nrmse(y_train, y_pred_train)
    nrmse_test  = compute_nrmse(y_test,  y_pred_test)
    print(f"NRMSE (train): {nrmse_train:.4f}")
    print(f"NRMSE (test) : {nrmse_test:.4f}")
    
    regression_results.append({
        'Dataset': ds_name,
        'Train NRMSE': f"{nrmse_train:.4f}",
        'Test NRMSE': f"{nrmse_test:.4f}",
        'Alpha': best_alpha
    })

### Regression Results Summary Table

In [ ]:
print("\n" + "="*70)
print("REGRESSION RESULTS")
print("="*70)
df_reg = pd.DataFrame(regression_results)
print(df_reg.to_string(index=False))

---
# Summary: All New Concepts You Need to Learn
---

Here is everything this notebook uses that was **NOT** taught in Lab 3:

| # | New Concept | Why It's Needed | Lab 3 Equivalent |
|---|---|---|---|
| 1 | **`RidgeClassifierCV`** | Auto-selects best alpha via cross-validation for classification | Lab 3 uses `LogisticRegression` with manual C tuning |
| 2 | **`RidgeCV`** | Auto-selects best alpha via cross-validation for regression | Lab 3 uses `Ridge` with manual alpha loop |
| 3 | **`LabelEncoder`** | Converts string labels → integers | Lab 3 datasets had numeric labels already |
| 4 | **`np.unique()`** | Find unique class labels and their counts | Lab 3 never explored class composition |
| 5 | **NRMSE metric** | `sqrt(MSE) / std(y)` — normalized, comparable across datasets | Lab 3 only uses raw MSE |
| 6 | **`class_weight='balanced'`** | Handles imbalanced classes by upweighting rare ones | Lab 3 never discusses class imbalance |
| 7 | **`MinMaxScaler` (fit/transform)** | Proper normalization that doesn't leak test info | Lab 3 uses `minmax_scale()` on all data at once |
| 8 | **`np.load()`** | Load `.npy` binary files | Lab 3 uses `pd.read_csv()` or sklearn built-ins |
| 9 | **`plt.bar()`** | Bar charts for class distribution | Lab 3 only uses scatter and line plots |
| 10 | **Target statistics (mean, var)** | Understand your regression target before modeling | Lab 3 doesn't explore target distributions |

---

### What's NOT Covered Here (Because It's NOT in Lab 3)

The project also asks for **Neural Network variants** (PyTorch/TensorFlow). That requires concepts like:
- Defining network layers, optimizers (Adam, AdamW, NAdam)
- BatchNormalization, activation functions
- DataLoaders for mini-batch gradient descent
- Different loss functions for binary vs. multiclass classification

These are covered in future labs, not Lab 3.